# PathSnap: quick EDA and MLP run (Colab)

This notebook can run in quick-sample or full-dataset mode. It reads only `train` and `student_test`; it never accesses evaluator trips. Set `FULL_DATASET = True` in the configuration cell to inspect all trips and train/evaluate using all 80 train and 10 student-test trips. The EDA cell skips full road-graph connectivity, which is expensive. The default small MLP run is a smoke check; **do not report its scores as final experiment results**.

Upload the three current source files when prompted, or place them in Drive and set `SOURCE_DIR` below. Keep your dataset archive in Drive so you do not upload it to GitHub.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os

# Change this to the folder containing train/ and student_test/.
DATA_ROOT = Path('/content/drive/MyDrive/PathSnap/datasets')
# Optional folder containing data_processing/candidate_features.py and models/mlp.py
SOURCE_DIR = Path('/content/drive/MyDrive/PathSnap/source')
print('Data root exists:', DATA_ROOT.exists())
print('Partitions:', [p.name for p in DATA_ROOT.iterdir()] if DATA_ROOT.exists() else 'Set DATA_ROOT first')


In [ ]:
# False = quick sample; True = all 80 train + 10 student_test trips
FULL_DATASET = False


## Fast EDA sample

The next cell inspects only a few permitted trips. It reports observation/node/arc/route counts and timestamp gaps, then plots GPS tracks without drawing millions of road edges or computing graph components. Raise `TRIPS_PER_PARTITION` if you want more coverage.


In [ ]:
import csv, math, statistics
import matplotlib.pyplot as plt

TRIPS_PER_PARTITION = None if FULL_DATASET else 3

def read_rows(path):
    with open(path) as f:
        return [line.split() for line in f if line.strip()]

summary=[]
plots_shown=0
for partition in ('train','student_test'):
    part=DATA_ROOT/partition
    all_trips=sorted(p for p in part.iterdir() if p.is_dir())
    trips=all_trips if TRIPS_PER_PARTITION is None else all_trips[:TRIPS_PER_PARTITION]
    for folder in trips:
        tid=folder.name
        track=read_rows(folder/f'{tid}.track')
        nodes=read_rows(folder/f'{tid}.nodes')
        arcs=read_rows(folder/f'{tid}.arcs')
        route=read_rows(folder/f'{tid}.route')
        ts=[float(r[2]) for r in track]
        gaps=[b-a for a,b in zip(ts,ts[1:])]
        summary.append({'partition':partition,'trip_id':tid,'gps_rows':len(track),
                        'nodes':len(nodes),'arcs':len(arcs),'route_arcs':len(route),
                        'gaps_gt_1s':sum(g>1 for g in gaps),'max_gap_s':max(gaps,default=0)})
        if not FULL_DATASET or plots_shown < 3:
            lon=[float(r[0]) for r in track]; lat=[float(r[1]) for r in track]
            plt.figure(figsize=(5,5)); plt.plot(lon,lat,'.-',markersize=2,linewidth=.6)
            plt.title(f'{partition} {tid}: GPS only'); plt.xlabel('longitude'); plt.ylabel('latitude'); plt.axis('equal'); plt.show()
            plots_shown += 1
print('Sampled EDA:')
for row in summary: print(row)


## Optional: MLP smoke run on a small subset

Upload `candidate_features.py`, `mlp.py`, and `knn.py` from your current `feature/member1-mlp` branch if they are not in `SOURCE_DIR`. This cell makes temporary train/student folders with a few trips, then invokes the real raw-file pipeline. Choose at least 3 training trips for 3-fold CV. Increase the subset and use the full documented settings only for a real experiment after checking alignment and candidate misses.


In [ ]:
# Copy the two source files into the expected package paths.
# If the files are already in SOURCE_DIR, this uses them; otherwise Colab will prompt for upload.
from pathlib import Path
import shutil, sys

repo=Path('/content/pathsnap-src')
(repo/'data_processing').mkdir(parents=True,exist_ok=True)
(repo/'models').mkdir(parents=True,exist_ok=True)
expected={
 'candidate_features.py': repo/'data_processing/candidate_features.py',
 'mlp.py': repo/'models/mlp.py',
 'knn.py': repo/'models/knn.py',
}
for name,dst in expected.items():
    src=SOURCE_DIR/name
    if src.exists(): shutil.copy2(src,dst)
missing=[name for name,dst in expected.items() if not dst.exists()]
if missing:
    from google.colab import files
    uploaded=files.upload()
    for name,data in uploaded.items():
        if name in expected: expected[name].write_bytes(data)
if any(not dst.exists() for dst in expected.values()):
    raise FileNotFoundError('Upload candidate_features.py and mlp.py, or set SOURCE_DIR correctly')
sys.path.insert(0,str(repo))


In [ ]:
# Select a small development subset. This uses only train/student_test folders.
import os, shutil, subprocess

QUICK_TRAIN_TRIPS = 80 if FULL_DATASET else 5  # >= 3 for CV
QUICK_STUDENT_TRIPS = 10 if FULL_DATASET else 2
FOLDS = 5 if FULL_DATASET else 3
EPOCHS = 40 if FULL_DATASET else 5
quick=Path('/content/pathsnap-quick-data')
for part,count in [('train',QUICK_TRAIN_TRIPS),('student_test',QUICK_STUDENT_TRIPS)]:
    dst=quick/part
    dst.mkdir(parents=True,exist_ok=True)
    folders=sorted(p for p in (DATA_ROOT/part).iterdir() if p.is_dir())[:count]
    for folder in folders:
        link=dst/folder.name
        if link.exists() or link.is_symlink():
            if link.is_symlink() or link.is_file(): link.unlink()
            else: shutil.rmtree(link)
        link.symlink_to(folder, target_is_directory=True)
    print(part, 'selected trips:', [p.name for p in folders])

cmd=[sys.executable,str(repo/'models/mlp.py'),
     '--train-root',str(quick/'train'),'--student-root',str(quick/'student_test'),
     '--out','/content/pathsnap-quick-results','--folds',str(FOLDS),'--epochs',str(EPOCHS)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)
print(Path('/content/pathsnap-quick-results/metrics.json').read_text())


## Optional: KNN smoke run

KNN can also be slow on large candidate tables. This runs the same selected trips with a small neighbor count; it is a pipeline check, not reportable evaluation. Colab includes scikit-learn in most runtimes.


In [ ]:
knn_cmd=[sys.executable,str(repo/'models/knn.py'),
          '--train-root',str(quick/'train'),'--student-root',str(quick/'student_test'),
          '--out','/content/pathsnap-knn-quick-results','--folds',str(FOLDS),'--neighbors','5']
print('Running:', ' '.join(knn_cmd))
subprocess.run(knn_cmd,check=True)
print(Path('/content/pathsnap-knn-quick-results/metrics.json').read_text())


## Outputs and limits

Quick EDA is printed/displayed in this notebook. The small MLP run writes diagnostics under `/content/pathsnap-quick-results`; download or copy them to Drive if you need to inspect them later. Colab's temporary `/content` storage is not permanent.

Before using results in the report: review the monotonic route-alignment assumption, inspect candidate miss rates, rerun with all permitted training trips and the chosen full configuration, and evaluate on all 10 student-test trips. Keep evaluator trips untouched. The notebook intentionally does not read them.
